# Lab - Unit 13: race day on the real car

**Runs on your machine with CARLA 0.9.12 and `ROAR_PY`.**

On racekit, Unit 13 assembled one file for one known track, re-tuned it against cars it had never met, kept a margin on every knob at its edge, and checked it before the race.
The real test of all of that is CARLA's car.
**In this lab you race your own race-day file on CARLA, the way the official race will**: one file, five official-runner launches in a row, none discarded, and a record you can defend.

What you do, in order:

1. name your **starting file**: the latest configuration that has already raced clean on CARLA (§2);
2. **assemble** the race-day file from it: a margin on every knob that reached an edge on CARLA, and `MPC_MIN_HORIZON` equal to its `MPC_HORIZON` (§3);
3. check its **timing** on your own machine, off CARLA (§3, step 1);
4. **accept** it with two clean practice launches, or race the starting file instead (§3, step 2);
5. go through the **checklist**, and write down the file's SHA-256 (§4);
6. race the **race-day record**: five consecutive launches of that one file (§5), each analysed and typed into the record (§6, §7);
7. never commit your results.

How to use it:

- The cells tagged `no-exec` are for **your machine** (the Windows PC you set up in the Unit 1 lab).
  Their shell lines are shown with `!` only so the notebook stays valid.
- The analysis cells run anywhere.
  In CI they run against a racekit-made sample recording (the starter on `monza-like`, with no respawn), so the book always proves they work; on your machine they run against your own recordings, **one at a time**.

> **Never commit your results.**
> This repository is public.
> Your recordings (the telemetry CSVs and any `.sections.txt` sidecars), screenshots and logs stay on your own machine, **outside the repository**
> (the Unit 1 lab uses `%USERPROFILE%\roar-runs\`).
> **Do not commit this notebook after you have typed your results into it** (§7): the typed-in launches are your data.
> The repository ignores data files under `labs/`, and a CI test fails if anything other than a notebook
> or a README is ever tracked there, but the first line of defence is not putting them here at all.

In [ ]:
import os

TELEMETRY_PATH = os.environ.get("ROAR_LAB_CSV", "telemetry.csv")

The parameters cell above is the **only** place a recording's file name appears.
Set `ROAR_LAB_CSV` to one recording's full path (outside the repository) before you start Jupyter; the recorder writes to the same variable.
Every launch of this lab is its own recording, so give each its own name (the commands in §5 do) and point `ROAR_LAB_CSV` at the one you are analysing.

## 1. The five steps

The Unit 1 lab (`units/unit-01-the-race-and-baseline/labs/real-sim.ipynb`) has every detail; this lab only changes what you race, and how many times.

1. **Start the Monza v1.1 executable** (CARLA 0.9.12) and wait for the map to load (Unit 1 lab, §1 for the pinned versions and §3 for the start).
2. **Put the submission in the forked competition folder**: the file §3 assembles, with the Unit 1 recorder and its three wiring lines, Unit 5's one-line `offset` change and Unit 7's section sidecar pasted in, exactly as for your earlier labs.
   Before you go to CARLA, race the edited file on racekit and run `check_submission` on it, as the Unit 1 lab says (§3, step 1, for a file with a Monza line).
3. **Run `competition_runner.py`** from `ROAR_Competition\competition_code`, with `ROAR_LAB_CSV` set to a new file outside the repository.
   Note the time and the collision count the runner reports, and whether the race finished.
4. **Export the telemetry to CSV**: the recorder does it as the car drives, one row per tick, to `ROAR_LAB_CSV`.
5. **Analyse it**: start Jupyter with `ROAR_LAB_CSV` pointing at the recording and run §6 onwards of this notebook.

**The `offset` column is the error this lab judges.**
Since Unit 5's change the recorder writes the agent's own cross-track error, `self.cross_track_error`, measured from its **own path**: the racing line when `LINE_CONTROL` is set, the centreline when it is `None`.
So the column is already the line-relative error of the clean rule.

## 2. The starting file

**Nothing that has not raced clean on CARLA is ever the race-day file.**
So race day starts from the latest configuration that has, taken in this order:

1. the **Unit 10 lab's** result, if it accepted one (MPC accepted: clean on R1 and R2, or on R3 and R4);
2. else the **Unit 8 lab's** accepted setting (clean on both of its races);
3. else the **Unit 7 lab's** accepted submission (clean on both of its re-races; a PROVISIONAL result that the Unit 10 lab's R00a-R00b raced clean twice counts too);
4. else the file you raced clean in the **Unit 5 or the Unit 1 lab**.

If none of them has ever raced clean on CARLA, there is no starting file: this lab stops, and the Unit 1 and Unit 7 labs come first.

Write down which lab your starting file came from, its exact constants, and its clean race times: they are its **record**.
The starting file is also your **fallback**: whatever happens below, it can race on race day.

## 3. Assemble, time, and accept

**Assemble** the race-day file from the starting file's **own** constants: its line (`LINE_CONTROL`, or `None`), its grip (`MU_G`, or `None` with `A_LAT` and `GRIP`), its `MAX_STEER`, its fraction (`MU_FRACTION`) and its steering choice (`STEERING`).
Then change exactly two things:

- **A margin on every knob that reached an edge on CARLA.**
  A knob reached an edge when one of your labs raced it one grid step higher and that race was not clean (the Unit 8 lab's ladder, or the Unit 10 lab's R3-R4 for `MU_FRACTION`).
  Its edge is the highest value that raced clean; the race-day file uses one grid step inside it (0.02 for `MU_FRACTION`; for any other knob, the grid step your lab raced it on), as the lesson's §3 did on racekit.
  **A knob never pushed to a setting that was not clean gets no margin**: you have not found its edge, so there is nothing to step back from.
- **`MPC_MIN_HORIZON` equal to the file's own `MPC_HORIZON`** (40, or 20 if the Unit 10 lab's step 1 chose 20), if the file has one (Unit 10's file and later).
  The horizon is then never cut during the race, so the race-day launches race exactly the behaviour the practice launches tested.
  Everything else stays as the starting file has it, `MPC_BUDGET_MS = 5.0` included: a slow call is still counted, and the timing is checked before the race instead (step 1).

If no knob reached an edge and the file has no MPC, the assembled file **is** the starting file: skip step 2.

**Step 1: the timing, off CARLA, first.**
Run `check_submission` on the assembled file, on the machine that will run the agent, with nothing else running (the Unit 10 lab's step 1).
The file carries a Monza line, which racekit's `test-circuit` refuses, so check its **timing copy**: the same file with `LINE_CONTROL = None` and nothing else changed (check that with a file comparison), and accept it only with a p99 of at most **4 ms**, because the copy's timing is a proxy for the race file's.
A file whose own `LINE_CONTROL` is `None` is checked as it is, at 5 ms (`HAS_LINE = False` in the cell below; the cell checks at 4 ms unless you say so).
With `MPC_MIN_HORIZON = MPC_HORIZON` the file can no longer shorten its horizon on a slow machine: **this check is its only timing protection**, so never skip it and never run it on a busy machine.
If it fails, the assembled file is not raced and there are no practice launches: the **starting file** is the race-day file.
Run the same check on the starting file too, and write its result down; if it fails, the machine is slow for that file as well, and that is a result to write down, but the starting file still races: it has already raced clean on CARLA on this machine, and it keeps its own horizon policy (only the assembled file fixes the horizon).

**Step 2: acceptance by two practice launches.**
If the assembled file differs from the starting file, it has not raced on CARLA yet, so it must be **accepted** first, by the earlier labs' rule: two practice launches (P1, P2), **both clean** by §6's rule.
Both clean: the assembled file is the **race-day file**.
Either not clean: the **starting file** is the race-day file, unchanged.
A practice launch with a host failure (§5) says nothing about the file: it is recorded, not counted, and launched again.
The practice launches are not part of the race-day record.

In [ ]:
# On your machine, in this notebook, with the CARLA server NOT running and nothing else busy. Not run in CI.
# Step 1 comes first: time the assembled file before any practice launch.
from pathlib import Path

from racekit.check_submission import check_submission

RACE_FILE = Path(r"C:\path\to\roar-runs\race-day\submission.py")  # the timing copy (the file itself only if its LINE_CONTROL is None)
# Fail closed: a file with a Monza LINE_CONTROL is checked through its timing copy (LINE_CONTROL = None), a proxy
# that passes only at p99 <= 4 ms. Set HAS_LINE = False only for a file whose own LINE_CONTROL is None (5 ms).
HAS_LINE = True
report = check_submission(RACE_FILE)
print(report)
limit_ms = 4.0 if HAS_LINE else 5.0
passed = report.ok and report.step_p99_ms is not None and report.step_p99_ms <= limit_ms
print(f"CHECK passed (p99 at most {limit_ms} ms):", passed)

In [ ]:
# Anaconda Prompt, on the Windows machine, from ROAR_Competition\competition_code. Not run in CI.
# Type each line WITHOUT the leading "!" (it is there only so the notebook stays valid).
# Step 2, only after step 1's timing check (the cell above) passed.
# 1. The assembled file: your starting file with the margin and MPC_MIN_HORIZON changed (section 3), and the
#    recorder, Unit 5's change and the sidecar pasted in, as submission.py
!copy %USERPROFILE%\roar-runs\race-day\submission.py submission.py
# 2. The practice launches P1 and P2, one recording each (only if the assembled file differs from the starting file)
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-13-p1.csv" && python competition_runner.py
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-13-p2.csv" && python competition_runner.py
# 3. Analyse one recording: start Jupyter with the variable pointing at it (from your clone of this repository)
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-13-p1.csv" && jupyter lab

## 4. The checklist

Before the race-day record, check the race-day file (the lesson's §6).
If the **starting file** is racing (§3), it races exactly as its earlier lab raced it: the items about the margin, the horizon and a passed timing check apply only to an assembled file, and for the starting file you write down its own step-1 result, passed or not.

- **one file**, `submission.py`, in the official shape, importing only the standard library, numpy and `roar_py_interface`;
- **Python 3.8**: `check_submission` built and raced it (§3, step 1), and the official environment is Python 3.8;
- **the runner files unmodified**: `competition_runner.py` and `infrastructure.py` exactly as the fork has them (`git status` in your fork shows only `submission.py` changed);
- **the season's wording**: open the current season's page and read what it says about extra dependencies, `instruction.txt` or `requirements.txt` (the series page and the gitbook disagree); write down which, and the date you read it;
- **every changed constant on its grid and written with `repr`**, with a comment saying why (the margin, the horizon);
- for a file with MPC (Unit 10's file and later): **`MPC_BUDGET_MS = 5.0`** as shipped, and, for an assembled file, `MPC_MIN_HORIZON` equal to `MPC_HORIZON`;
- **the timing**: an assembled file passed step 1 on this machine, idle, and its practice launches accepted it (§3, step 2); or the starting file is racing, unchanged, with its own step-1 result written down;
- **the file's SHA-256**, from the operating system's own tool (the cell below), written down with the date: the five launches must all race that exact file.

The race-day record below races **one file**: if anything in it changes after you write the hash down, the record starts again.

In [ ]:
# Anaconda Prompt, from ROAR_Competition\competition_code. Not run in CI.
# Print the SHA-256 of the race-day file, and write it down; print it again after the fifth launch: the same.
!certutil -hashfile submission.py SHA256

## 5. The race-day record

**Five consecutive official-runner launches of the race-day file.**
None is discarded and none is restarted because it was slow or crashed: a crash is part of the record, as it would be in the real race.
Analyse each recording (§6) and type its result into `LAUNCHES` (§7) before the next launch, so the record is written as it happens.

**A host failure is not an agent failure.**
If the CARLA server exits, or the agent never receives its first tick, the launch says nothing about your file.
Record it (`host_failure = True` in its row), then **start a new block of five**: launch 1 again.
The record keeps both blocks, the interrupted one and the new one, and the result is the last **complete** block (design 013 §7.3: the author's own qualification follows the same rule).

In [ ]:
# Anaconda Prompt, from ROAR_Competition\competition_code. Not run in CI.
# Five launches of the same submission.py, one recording each; after a host failure, a new block: b2-l1, b2-l2, ...
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-13-b1-l1.csv" && python competition_runner.py
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-13-b1-l2.csv" && python competition_runner.py
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-13-b1-l3.csv" && python competition_runner.py
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-13-b1-l4.csv" && python competition_runner.py
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-13-b1-l5.csv" && python competition_runner.py
# Analyse each one in turn: restart Jupyter with ROAR_LAB_CSV pointing at it, and run sections 6 and 7
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-13-b1-l1.csv" && jupyter lab

## 6. Analyse one launch

Set `RACE_FINISHED` and `RUNNER_COLLISIONS` to what the runner reported for this launch (Unit 1 lab, §4, explains why the recorder cannot know).
The cell applies the book's **clean rule** to the recording, as the Unit 10 lab does: the race finished, with 0 collisions and no respawn, and the max |cross-track error| from the agent's own path (the `offset` column) is at most 2 m over the driving rows (`held == 0`), without the stale first row after a hold.
If the race is complete, it also prints the race time, from the sector splits as the Unit 1 lab computes it.
Type this launch's row into `LAUNCHES` in §7 from what it prints.

In [ ]:
from math import isnan

import matplotlib.pyplot as plt
import numpy as np
from racekit.telemetry import read_csv, sector_splits

RACE_FINISHED = True  # set False if the runner reported that the race did not finish
RUNNER_COLLISIONS = 0  # the collision count the runner reported
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the max |cross-track error| from the agent's own path


def driving_rows(telemetry):
    # The rows the clean rule judges: not held, and not the stale first row after a hold run.
    held = telemetry["held"] == 1
    keep = ~held
    stale = [row + 1 for row in np.nonzero(held)[0] if row + 1 < len(held) and not held[row + 1]]
    keep[np.array(stale, dtype=int)] = False
    return keep


def clean_check(telemetry):
    # (clean, respawns, max |error| in m) by the book's rule
    held = telemetry["held"] == 1
    respawns = len([row for row in np.nonzero(held)[0] if row == 0 or not held[row - 1]])
    keep = driving_rows(telemetry)
    worst = float(np.max(np.abs(telemetry["offset"][keep]))) if np.sum(keep) else float("nan")
    is_clean = RACE_FINISHED and RUNNER_COLLISIONS == 0 and respawns == 0 and worst <= CLEAN_ERROR
    return is_clean, respawns, worst


def race_time(telemetry, meta):
    # The race time from the sector splits (Unit 1 lab), or None when a sector was never reached.
    splits = sector_splits(telemetry, meta["progress_total"], laps=meta["laps"], per_lap=meta["per_lap"],
                           finished=RACE_FINISHED)
    if any(isnan(split) for lap in splits for split in lap):
        return None
    held_rows = np.nonzero(telemetry["held"] == 1)[0]
    restart = float(telemetry["t"][held_rows[-1]]) if len(held_rows) else 0.0  # after the last hold
    return restart + float(sum(sum(lap) for lap in splits))


telemetry, meta = read_csv(TELEMETRY_PATH)
is_clean, respawns, worst = clean_check(telemetry)
this_time = race_time(telemetry, meta)
print(f"this launch: {len(telemetry)} rows, {respawns} respawns, "
      f"max |cross-track error from its own path| {worst:.3f} m")
print(f"clean (finished, 0 collisions, no respawn, max |e| <= {CLEAN_ERROR} m): {is_clean}")
print("race time: " + ("not complete" if this_time is None else f"{this_time:.2f} s"))
shown_time = "None" if this_time is None else f"{this_time:.2f}"
print(f"its row for LAUNCHES: ({shown_time}, {RUNNER_COLLISIONS}, {RACE_FINISHED}, {worst:.3f}, {is_clean}, False)")
if worst == 0.0:
    print("every offset is 0: was this recorded without Unit 5's one-line change?")

In [ ]:
keep = driving_rows(telemetry)
fig, (ax_error, ax_speed) = plt.subplots(1, 2, figsize=(12, 3.5))
ax_error.plot(telemetry["t"][keep], telemetry["offset"][keep], linewidth=0.6)
for limit in (-CLEAN_ERROR, CLEAN_ERROR):
    ax_error.axhline(limit, color="grey", linestyle="--", linewidth=1.0)
ax_error.set_xlabel("t (s)")
ax_error.set_ylabel("cross-track error (m)")
ax_error.set_title("the error from the agent's own path, and the 2 m rule (dashed)")
ax_speed.plot(telemetry["t"], telemetry["v"], linewidth=0.6)
ax_speed.set_xlabel("t (s)")
ax_speed.set_ylabel("speed (m/s)")
ax_speed.set_title("speed")
fig.tight_layout()
plt.show()

**What racekit predicts** (`agents/unit-13.py` on `monza-like`, the carla-like car; the milestone and the routed premise assert it): 468.55 s on the nominal car (race seeds 0-4, every race clean, line error at most 1.01 m), clean on every perturbed car it was checked on, against `agents/unit-07.py`'s 515.10 s.
racekit is not CARLA, `monza-like` is not CARLA's Monza, and your race-day file is your own, from your own labs: compare the **pattern** (is every launch clean, and is the file faster than your Unit 7 record by Unit 1's rule?), not the second decimal.

## 7. The record

Type every launch into `LAUNCHES`, in the order you launched them, as `(time, runner collisions, finished, max error, clean, host failure)`:

- `time`: §6's race time in s, or `None` if it was not complete;
- `runner collisions` and `finished`: what the runner reported;
- `max error`: §6's max |cross-track error| in m;
- `clean`: §6's verdict;
- `host failure`: `True` only for a host failure (then the other fields may be `None`).

Type your **Unit 7 lab record** into `UNIT_7_TIMES`: the clean race times of the submission the Unit 7 lab accepted (or your Unit 1 lab times if that is your only clean record).

The summary cell groups `LAUNCHES` into five-launch **blocks** (a host failure ends its block, and the next launch starts a new one), keeps every block, and reports the last complete one: its clean count, the median and spread of its clean launches, and Unit 1's comparison with your Unit 7 record.

In [ ]:
from racekit.measure import noise_aware_better

BLOCK = 5  # the race-day record: five consecutive launches of one file
REASONABLE = 329.0  # the book's bar (Unit 1 lab): every launch clean, median at most this
# Your launches, in order: (time in s or None, runner collisions, finished, max error in m, clean, host failure).
LAUNCHES = []
UNIT_7_TIMES = []  # your Unit 7 lab record: its clean race times (s)


def blocks_of(launches):
    """Split the launches into blocks: a block ends after five launches, or at a host failure (which stays in it).
    Returns a list of (launches, status), status "complete", "host failure" or "in progress"."""
    blocks, current = [], []
    for launch in launches:
        if len(launch) != 6:
            raise ValueError(f"a launch has 6 fields, not {len(launch)}: {launch}")
        seconds, collisions, finished, error, clean, host_failure = launch
        current.append(launch)
        if host_failure:
            blocks.append((current, "host failure"))
            current = []
            continue
        expected = bool(finished) and collisions == 0 and error is not None and error <= CLEAN_ERROR
        if clean and not expected:
            raise ValueError(f"launch {launch} is typed clean, but the clean rule says it is not: check the row")
        if len(current) == BLOCK:
            blocks.append((current, "complete"))
            current = []
    if current:
        blocks.append((current, "in progress"))
    return blocks


def summary(launches, unit_7_times):
    """(the blocks, the last complete block's summary or None)."""
    blocks = blocks_of(launches)
    statuses = [status for _, status in blocks]
    if "complete" in statuses and statuses.index("complete") != len(statuses) - 1:
        raise ValueError("a launch after a complete block: the race-day record is one block of five launches")
    complete = [block for block, status in blocks if status == "complete"]
    if not complete:
        return blocks, None
    block = complete[-1]
    times = [launch[0] for launch in block if launch[4]]
    result = {"clean": len(times), "launches": len(block), "times": times,
              "median": float(np.median(times)) if times else None,
              "spread": max(times) - min(times) if times else None,
              "beats_unit_7": noise_aware_better(times, unit_7_times) if times and unit_7_times else None}
    return blocks, result


# Worked examples (made up, not recordings).
five_clean = [(330.2, 0, True, 1.4, True, False), (329.9, 0, True, 1.3, True, False),
              (330.5, 0, True, 1.5, True, False), (330.1, 0, True, 1.4, True, False), (330.0, 0, True, 1.3, True, False)]
blocks, result = summary(five_clean, [352.4, 351.8])
assert len(blocks) == 1 and result["clean"] == 5 and abs(result["median"] - 330.1) < 1e-9 and result["beats_unit_7"]
# A host failure at launch 3: that block is kept, and the next five launches are the record.
host = [(None, None, None, None, False, True)]
blocks, result = summary(five_clean[:2] + host + five_clean, [352.4, 351.8])
assert [status for _, status in blocks] == ["host failure", "complete"] and result["clean"] == 5
# A crash is part of the record, never discarded: 4 of 5 clean, and the crash's time is not in the median.
crash = (612.3, 1, True, 6.2, False, False)
blocks, result = summary(five_clean[:2] + [crash] + five_clean[2:4], [352.4, 351.8])
assert result["clean"] == 4 and result["launches"] == 5 and 612.3 not in result["times"]
# A row typed clean that the rule says is not, and any launch after a complete block (a sixth, or a second block), are refused.
for bad in ([(330.2, 1, True, 1.4, True, False)], five_clean + five_clean[:1], five_clean + five_clean[:1] + five_clean[:4]):
    try:
        summary(bad, [])
    except ValueError:
        pass
    else:
        raise AssertionError("a bad record was accepted")
# Nothing typed yet: no blocks, no result.
assert summary([], []) == ([], None)

blocks, result = summary(LAUNCHES, UNIT_7_TIMES)
for number, (block, status) in enumerate(blocks, start=1):
    print(f"block {number} ({status}): {len(block)} launches, {sum(1 for launch in block if launch[4])} clean")
if result is None:
    print(f"no complete block of {BLOCK} launches yet: {len(LAUNCHES)} launches typed in")
else:
    print(f"the race-day record (the last complete block): {result['clean']}/{result['launches']} clean")
    if result["times"]:
        print(f"clean launches: median {result['median']:.2f} s, spread {result['spread']:.2f} s")
        verdict = result["clean"] == result["launches"] and result["median"] <= REASONABLE
        print(f"every launch clean and the median at most {REASONABLE:.0f} s (the book's bar): {verdict}")
    if result["beats_unit_7"] is None:
        print("Unit 1's comparison needs UNIT_7_TIMES and at least one clean launch")
    else:
        print(f"Unit 1's rule: the race-day file beats your Unit 7 record ({len(UNIT_7_TIMES)} races): "
              f"{result['beats_unit_7']}")

## What to take away

- On race day, **only a file that has raced clean on CARLA races**: the starting file is the fallback, and an assembled file earns its place with two clean practice launches.
- A margin goes on **every knob that reached an edge**, and only those: a knob you never pushed has no edge to step back from.
- The race-day file races **the behaviour you tested**: its horizon never changes mid-race, and its timing is checked on the race machine, idle, before the race.
- The record is **five consecutive launches**, none discarded; a host failure restarts the block and both blocks are kept.

> **One last check: never commit your results.**
> This repository is public.
> Your recordings, screenshots and logs stay on your own machine, in the folder outside the repository
> that `ROAR_LAB_CSV` pointed at, and so does this notebook once your launches are typed into it.
> Run `git status` in your clone before you commit anything: none of them should be listed.